# Tokenizers

For this Colab session, we explore the world of Tokenizers

You can run this notebook on a free CPU, or locally on your box if you prefer.


## My practical focus for this notebook

I am keeping the Tokenizers theory from the course intact, but I am changing the examples so they connect with my own AI Engineering journey.

Today I want to understand:

- how text becomes tokens and token IDs
- how tokens become text again
- why different LLMs tokenize the same sentence differently
- how token counts affect LLM applications
- how chat templates convert `system/user/assistant` messages into the format a model expects
- why I cannot directly send Python dictionaries to an LLM
- how tokenizer behavior changes across general-purpose and coding models
- how this connects to the LLM applications I am building

**My main mental model:**

`Human Text → Tokenizer → Token IDs → LLM → Token IDs → Tokenizer → Human Text`


## Reminder: 2 important pro-tips for using Colab:

**Pro-tip 1:**

Don't worry about warnings and messages!

**Pro-tip 2:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [3]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [4]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

# Sign in to Hugging Face

1. If you haven't already done so, create a free HuggingFace account at https://huggingface.co and navigate to Settings, then Create a new API token, giving yourself write permissions

**IMPORTANT** when you create your HuggingFace API key, please be sure to select read/write permissions for your key by clicking on the WRITE tab, otherwise you may get problems later.

2. Press the "key" icon on the side panel to the left, and add a new secret:
`HF_TOKEN = your_token`

3. Execute the cell below to log in.

In [5]:
# Log in to Hugging Face

hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

# Check Google Colab GPU

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

HF key looks good so far
Wed Aug 26 16:06:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------

# Loading a tokenizer

I will start with a tokenizer from a model that is practical for this notebook.

The important point is that I am loading the **tokenizer**, not the full LLM.

The tokenizer contains the rules needed to convert text into the token IDs expected by that model.

I am using Qwen2.5 because I will compare it later with Phi-4 and Qwen Coder.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Tokenizer loaded:", BASE_MODEL)

In [ ]:
text = (
    "I am learning AI Engineering and building practical "
    "LLM applications with Hugging Face."
)

tokens = tokenizer.encode(text)
tokens

In [ ]:
character_count = len(text)
word_count = len(text.split())
token_count = len(tokens)

print(f"Characters : {character_count}")
print(f"Words      : {word_count}")
print(f"Tokens     : {token_count}")

In [ ]:
decoded_text = tokenizer.decode(tokens)

print(decoded_text)

In [ ]:
# batch_decode is normally useful when decoding a batch of token sequences.
# Here I wrap our single sequence in a list.

decoded_batch = tokenizer.batch_decode([tokens])

print(decoded_batch[0])

In [ ]:
# Vocabulary entries that were added or customized for this tokenizer.
tokenizer.get_added_vocab()

In [ ]:
print("Vocabulary size:", len(tokenizer.vocab))

In [ ]:
len(tokenizer.vocab)

# Preparing a tokenizer for chat

I will use the same tokenizer and inspect its chat template.

The important distinction is:

**Chat messages are Python dictionaries.**

But the LLM ultimately processes **token IDs**.

`messages → apply_chat_template() → formatted conversation → token IDs → model`

The tokenizer knows the model-specific conversation format.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI Engineering mentor."
    },
    {
        "role": "user",
        "content": "Explain tokenization in simple language."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

## Crucial "Aha" moment

For several weeks, we have worked with APIs where we pass messages such as:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Explain AI Engineering simply"}
]
```

It is easy to imagine that the LLM directly understands those Python dictionaries.

**It does not.**

An LLM is a Data Science model that ultimately receives a sequence of numbers and predicts the probability of the next number.

So there is a transformation:

```text
Python dictionaries
        ↓
apply_chat_template()
        ↓
Model-specific chat text
        ↓
Tokenizer
        ↓
Token IDs
        ↓
LLM
        ↓
Generated token IDs
        ↓
Tokenizer.decode()
        ↓
Human-readable response
```

### This is why `apply_chat_template()` matters

Different chat models can use different special tokens and conversation formats.

`apply_chat_template()` applies the format that belongs to the selected tokenizer/model.

So I should **not manually invent chat special tokens** when the tokenizer already provides the correct template.

# Comparing Tokenizers Across Models

Now I will compare three models that are useful for understanding modern LLM workflows:

- **Phi-4 Mini Instruct** from Microsoft
- **DeepSeek-V3.1** from DeepSeek AI
- **Qwen2.5-Coder 7B Instruct** from Alibaba Cloud

The goal is not to decide which model is "best".

The goal is to understand an important AI Engineering fact:

> **The same sentence can produce different tokens and different token counts depending on the tokenizer used by the model.**

That matters for:
- context windows
- token usage
- latency
- API costs
- prompt design
- coding applications

In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Phi-4      :", PHI4)
print("DeepSeek   :", DEEPSEEK)
print("Qwen Coder :", QWEN_CODER)

In [ ]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = (
    "I am curiously excited to show Hugging Face tokenizers "
    "in action to my AI Engineering workflow."
)

base_tokens = tokenizer.encode(text)
phi_tokens = phi4_tokenizer.encode(text)

print("Qwen 2.5:")
print(base_tokens)
print("Decoded:", tokenizer.batch_decode([base_tokens]))

print("\nPhi-4:")
print(phi_tokens)
print("Decoded:", phi4_tokenizer.batch_decode([phi_tokens]))

print("\nToken counts:")
print("Qwen 2.5:", len(base_tokens))
print("Phi-4   :", len(phi_tokens))

In [ ]:
print("Qwen 2.5 chat template:")
print(
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
)

print("\nPhi-4 chat template:")
print(
    phi4_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
)

In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = (
    "I am curiously excited to show Hugging Face tokenizers "
    "in action to my AI Engineering workflow."
)

qwen_tokens = tokenizer.encode(text)
phi_tokens = phi4_tokenizer.encode(text)
deepseek_tokens = deepseek_tokenizer.encode(text)

print("Qwen 2.5 tokens     :", len(qwen_tokens))
print("Phi-4 tokens        :", len(phi_tokens))
print("DeepSeek tokens     :", len(deepseek_tokens))

print("\nRaw token IDs:")
print("Qwen     :", qwen_tokens)
print("Phi-4    :", phi_tokens)
print("DeepSeek :", deepseek_tokens)

In [ ]:
print("Qwen 2.5:")
print(tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
))

print("\nPhi-4:")
print(phi4_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
))

print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
))

In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)

code = '''
def greet_ai_engineer(name):
    print("Hello", name)
'''

tokens = qwen_tokenizer.encode(code)

print("Code:")
print(code)

print("\nToken → decoded piece")
for token in tokens:
    print(f"{token} = {qwen_tokenizer.decode([token])}")

# My Tokenizer Takeaways

## 1. Tokenizer

A tokenizer converts human-readable input into token IDs.

`Text → Tokens → Token IDs`

## 2. Decode

The reverse direction converts token IDs back into readable text.

`Token IDs → Text`

## 3. Token count

Token count is important because it affects:
- context-window usage
- inference workload
- latency
- API cost when pricing is token-based

## 4. Different models can tokenize differently

The same sentence can produce different token IDs and different token counts with different models.

## 5. Chat templates

`apply_chat_template()` converts:

```python
[
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."}
]
```

into the model's expected conversation format.

## 6. The complete LLM picture

```text
User message
     ↓
Chat messages
     ↓
apply_chat_template()
     ↓
Tokenizer
     ↓
Token IDs
     ↓
LLM
     ↓
Generated Token IDs
     ↓
Tokenizer.decode()
     ↓
Assistant response
```

### My AI Engineering takeaway

I now understand that an LLM application is not simply:

`Prompt → Model → Answer`

There is an important layer between my application and the model:

**tokenization + model-specific chat formatting.**

That understanding will be useful when I move from high-level Hugging Face pipelines to lower-level LLM inference and eventually build production AI applications.
